# 원판 YOLO26n · 100에폭과 최대 200에폭 학습 일정 비교

두 실행은 같은 사전학습 모델에서 출발한다. 기존 일정은 최대 100에폭·patience 100, 새 일정은 **최대 200에폭·patience 50**이다. 검증 fitness가 50에폭 연속으로 개선되지 않으면 일찍 끝날 수 있다. 총 에폭이 바뀌면 학습률 감소 일정과 마지막 10에폭 모자이크 해제 시점도 함께 바뀐다. 이 실험으로 각 변경의 효과를 따로 추정할 수는 없다.

먼저 14번에서 세 프레임의 허브를 판독한다. 이 좌표는 평가에만 쓴다. 모델 선택 기준은 검증 6명의 mAP50–95, 동작 중 정탐을 얻거나 잃은 프레임, 직접 판독한 허브의 위치 오차다.

In [ ]:
import os
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yaml
from IPython.display import display

os.environ["WANDB_MODE"] = "offline"  # 이 노트북의 학습 로그는 Mac에 저장한다.

from wlcv.annotation.hub_points import FRAMES, VIDEO, read_hubs
from wlcv.config import ROOT
from wlcv.training.plate_schedule import BASE, TRIAL, settings_for_trial, train_schedule
from wlcv.evaluation.schedule_review import compare_runs, evaluate_run, hub_summary
from wlcv.evaluation.schedule_tools import curves, fiftyone_comparison, hub_figure, log_scalar_history

control = yaml.safe_load((BASE / "args.yaml").read_text())
trial = settings_for_trial()
data = Path(trial["data"])
keys = ["model", "epochs", "patience", "seed", "batch", "imgsz", "optimizer", "lr0", "mosaic", "close_mosaic", "scale", "hsv_v"]
display(pd.DataFrame({"100": control, "200": trial}).loc[keys])
print("변경 설정", [key for key in control if control[key] != trial[key]])
print(f"허브 판독 {len(read_hubs())}/3장 · optimizer=auto가 고른 실제 값은 학습 시작 로그에서 확인")

## 1. Mac에서 새 일정 시작하기

세 장을 저장한 후 아래 값을 `True`로 바꾸고 셀을 실행한다. 베이스라인의 시작 모델·기기·증강 등 나머지 설정은 저장된 args.yaml에서 읽는다. 완료된 새 실행은 다시 열고, 중단돼 유효한 last.pt가 있으면 재개한다. 결과는 기존 베이스라인과 다른 폴더에 저장한다.

In [ ]:
START_TRAINING = False
if START_TRAINING:
    completed = train_schedule()
    print("완료", completed / "weights/best.pt", completed / "weights/last.pt", sep="\n")
else:
    print("학습 대기 · 준비되면 START_TRAINING을 True로 바꾸세요.")

## 2. 두 best.pt를 같은 입력에서 평가하기

mAP는 고정 검증 JPEG 39장 전체에서 Ultralytics 검증기로 다시 계산한다. 동작 중 비교는 여섯 검증 선수의 해당 원본 영상을 0번부터 동일하게 추론하고, 수동 박스가 있는 ready~finish 프레임만 정탐 여부를 비교한다. finish가 없는 하프 영상은 마지막으로 판독한 이벤트까지만 집계한다. 직접 검출(신뢰도 0.10)과 기존 시간 연결(후보 0.001, 반경 0.5배, 최대 2프레임)을 나눠 본다. 정탐 기준은 검수 IoU ≥ 0.50이다.

In [ ]:
ready = (TRIAL / "schedule_complete.txt").is_file()
runs = {"100": BASE, "200": TRIAL}
if ready:
    output = ROOT / "outputs/plate_schedule_comparison" / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
    output.mkdir(parents=True)
    metrics, records = [], {}
    for label, run in runs.items():
        score, detail = evaluate_run(run, data, output)
        records[label] = detail
        detail.to_csv(output / f"frames_{label}.csv", index=False)
        epochs = len(pd.read_csv(run / "results.csv"))
        metrics.append({"일정": label, "실제 에폭": epochs, "검증 mAP50–95": score})
    metrics = pd.DataFrame(metrics)
    metrics.to_csv(output / "metrics.csv", index=False)
    display(metrics.round(4))
    pair = compare_runs(records["100"], records["200"])
    pair.to_csv(output / "in_motion.csv", index=False)
    changes = pair[pair.change.isin(["복구", "손실"])]
    changes.to_csv(output / "recovered_lost.csv", index=False)
    display(pair.groupby(["video_id","method","change"]).size().unstack(fill_value=0))
    display(changes[["video_id","frame_index","method","change","source_100","source_200","iou_100","iou_200"]])
    curves(runs).savefig(output / "curves.png", dpi=140)
    plt.show()
else:
    print("새 일정이 완료된 후 이 셀부터 다시 실행하세요.")

## 3. 허브 위치와 검출 성공을 따로 보기

중심 오차는 모델 박스 중심에서 직접 찍은 허브까지 원본 픽셀 거리다. y 차이는 양수이면 모델 중심이 허브보다 아래다. 판독 불가·추정은 주 위치 평가에서 제외하고, 두 모델이 같은 프레임에서 모두 정탐인 쌍으로 중앙 오차를 비교한다. 정탐 수와 분모도 함께 확인한다.

In [ ]:
if ready:
    hubs = hub_summary(pair)
    hubs.to_csv(output / "hub_errors_by_video.csv", index=False)
    display(hubs.round(2))
    b02 = pair[pair.video_id.eq(VIDEO) & pair.frame_index.isin(FRAMES)]
    display(b02[["frame_index","method","visibility","match_100","match_200","dy_px_100","dy_px_200","error_px_100","error_px_200"]].round(2))
    hub_figure(pair).savefig(output / "b02_hubs.png", dpi=140)
    plt.show()
    print(output)

## 4. FiftyOne과 W&B에서 검수하기

FiftyOne의 고정 검증 사진 위에서 `ground_truth`, `model_100`, `model_200`을 켜고 끈다. Saved views의 Recovered·Lost는 검증 JPEG에서 직접 검출한 박스의 TP 변화다. 앞의 원본 연속 영상 표에는 시간 연결도 포함되어 있어 모집단이 다르다. W&B는 선택 사항이며 완료된 두 실행의 설정·손실·지표를 오프라인으로 기록한다. 사용하려면 Jupyter 실행 환경에 `wandb`가 필요하다.

In [ ]:
OPEN_FIFTYONE = False
LOG_WANDB_OFFLINE = False
if ready and OPEN_FIFTYONE:
    session = fiftyone_comparison(runs, data)
if ready and LOG_WANDB_OFFLINE:
    log_scalar_history(runs, output)

### 채택 기준

같은 검증 사람과 단계에서 복구·손실 목록을 먼저 확인한다. mAP50–95와 허브 오차가 같은 방향인지, 동작 중 새로 잃은 사례가 있는지 본다. b02 한 장이나 학습 손실만 좋아진 경우 기존 best.pt를 유지한다. 반복해서 비교한 검증셋에서의 한 번의 실행으로 기록하고 두 모델의 best.pt를 각 실행 폴더에 보관한다.